# 08. Model Benchmarking

## Objective

This notebook compares the baseline WOE Logistic Regression scorecard model against selected benchmark/challenger approaches.

Benchmarks included:

1. **WOE Logistic Regression on time split**  
   Official baseline model strategy from Notebook 07.

2. **WOE Logistic Regression with random stratified split**  
   Sensitivity benchmark to show how random validation can be more optimistic than time-based validation.

3. **XGBoost-style challenger on WOE features**  
   Tree-based challenger trained on the same final WOE features for a fair model-algorithm comparison.


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    brier_score_loss,
)
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Paths and input files

Inputs:

- final WOE datasets from Notebook 06;
- final WOE feature list from Notebook 06;
- baseline model outputs from Notebook 07, if available.

Outputs:

```text
data/processed/08.model_benchmarking/
```


In [ ]:
PROJECT_ROOT = Path("..")

WOE_INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06.woe_binning_iv_analysis"
BASELINE_INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "07.logistic_regression_scorecard"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "08.model_benchmarking"
CHARTS_DIR = OUTPUT_DIR / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE_CANDIDATES = [
    WOE_INPUT_DIR / "train_woe_final.pkl",
    WOE_INPUT_DIR / "train_woe.pkl",
]

VALIDATION_FILE_CANDIDATES = [
    WOE_INPUT_DIR / "validation_woe_final.pkl",
    WOE_INPUT_DIR / "validation_woe.pkl",
]

OOT_FILE_CANDIDATES = [
    WOE_INPUT_DIR / "oot_test_woe_final.pkl",
    WOE_INPUT_DIR / "oot_test_woe.pkl",
]

FEATURES_FILE_CANDIDATES = [
    WOE_INPUT_DIR / "final_candidate_woe_features.pkl",
    WOE_INPUT_DIR / "final_candidate_features.pkl",
    WOE_INPUT_DIR / "initial_selected_woe_features.pkl",
]


def first_existing_file(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Missing expected {label}. Checked: "
        + ", ".join(str(p) for p in candidates)
    )


TRAIN_FILE = first_existing_file(TRAIN_FILE_CANDIDATES, "train WOE file")
VALIDATION_FILE = first_existing_file(VALIDATION_FILE_CANDIDATES, "validation WOE file")
OOT_FILE = first_existing_file(OOT_FILE_CANDIDATES, "OOT WOE file")
FEATURES_FILE = first_existing_file(FEATURES_FILE_CANDIDATES, "feature list file")

print("Using input files:")
print(f"- Train:      {TRAIN_FILE}")
print(f"- Validation: {VALIDATION_FILE}")
print(f"- OOT:        {OOT_FILE}")
print(f"- Features:   {FEATURES_FILE}")

## 2. Load datasets and final WOE features

In [ ]:
train = pd.read_pickle(TRAIN_FILE)
validation = pd.read_pickle(VALIDATION_FILE)
oot_test = pd.read_pickle(OOT_FILE)

with open(FEATURES_FILE, "rb") as f:
    loaded_features = pickle.load(f)

TARGET_COL = "target_bad"

# Normalize feature names: supports both base names and *_woe names.
final_woe_features = []
for feature in loaded_features:
    if feature in train.columns and feature != TARGET_COL:
        final_woe_features.append(feature)
    elif f"{feature}_woe" in train.columns:
        final_woe_features.append(f"{feature}_woe")

final_woe_features = list(dict.fromkeys(final_woe_features))

if len(final_woe_features) == 0:
    available_woe_cols = [c for c in train.columns if c.endswith("_woe")]
    raise ValueError(
        "No final WOE features found. "
        f"Available WOE columns example: {available_woe_cols[:20]}"
    )

for sample_name, data in {
    "train": train,
    "validation": validation,
    "oot_test": oot_test,
}.items():
    if TARGET_COL not in data.columns:
        raise ValueError(f"{TARGET_COL} missing from {sample_name}")

print(f"Train shape:      {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"OOT shape:        {oot_test.shape}")
print(f"Final WOE features: {len(final_woe_features)}")

final_woe_features

## 3. Load baseline model results from Notebook 07

In [ ]:
baseline_report_file = BASELINE_INPUT_DIR / "logistic_regression_scorecard_report.xlsx"
baseline_artifacts_file = BASELINE_INPUT_DIR / "logistic_regression_scorecard_artifacts.pkl"

baseline_performance_from_nb07 = None

if baseline_report_file.exists():
    try:
        baseline_performance_from_nb07 = pd.read_excel(baseline_report_file, sheet_name="performance")
        print("Loaded baseline performance from Notebook 07 Excel report.")
        display(baseline_performance_from_nb07)
    except Exception as e:
        print(f"Could not load Notebook 07 performance sheet: {e}")
else:
    print("Notebook 07 Excel report not found. Baseline will be recalculated in this notebook.")

## 4. Common evaluation functions

In [ ]:
def gini_from_auc(auc):
    return 2 * auc - 1


def ks_statistic(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    ks_values = tpr - fpr
    idx = np.argmax(ks_values)

    return {
        "ks": ks_values[idx],
        "threshold": thresholds[idx],
        "tpr": tpr[idx],
        "fpr": fpr[idx],
    }


def evaluate_model_result(y_true, y_score, model_name, sample_name):
    auc = roc_auc_score(y_true, y_score)
    ks = ks_statistic(y_true, y_score)

    return {
        "model": model_name,
        "sample": sample_name,
        "rows": len(y_true),
        "bad_rate": np.mean(y_true),
        "auc": auc,
        "gini": gini_from_auc(auc),
        "ks": ks["ks"],
        "ks_threshold": ks["threshold"],
        "brier_score": brier_score_loss(y_true, y_score),
    }


def decile_report(y_true, y_score, model_name, sample_name, n_bins=10):
    temp = pd.DataFrame({
        "target": y_true,
        "pd_pred": y_score,
    })

    temp["risk_decile"] = pd.qcut(
        temp["pd_pred"].rank(method="first", ascending=False),
        q=n_bins,
        labels=list(range(1, n_bins + 1)),
    ).astype(int)

    out = (
        temp.groupby("risk_decile")
        .agg(
            accounts=("target", "size"),
            bads=("target", "sum"),
            goods=("target", lambda x: (x == 0).sum()),
            observed_bad_rate=("target", "mean"),
            avg_predicted_pd=("pd_pred", "mean"),
            min_predicted_pd=("pd_pred", "min"),
            max_predicted_pd=("pd_pred", "max"),
        )
        .reset_index()
    )

    out.insert(0, "sample", sample_name)
    out.insert(0, "model", model_name)

    out["cum_bads"] = out["bads"].cumsum()
    out["cum_accounts"] = out["accounts"].cumsum()
    out["cum_bad_capture_rate"] = out["cum_bads"] / out["bads"].sum()
    out["cum_population_rate"] = out["cum_accounts"] / out["accounts"].sum()

    return out


def plot_roc_comparison(prediction_table, sample_name, charts_dir):
    subset = prediction_table[prediction_table["sample"] == sample_name].copy()

    fig, ax = plt.subplots(figsize=(8, 6))

    for model_name in subset["model"].unique():
        model_subset = subset[subset["model"] == model_name]
        fpr, tpr, _ = roc_curve(model_subset["target"], model_subset["pd_pred"])
        auc = roc_auc_score(model_subset["target"], model_subset["pd_pred"])
        ax.plot(fpr, tpr, linewidth=2.3, label=f"{model_name} | AUC={auc:.4f}")

    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC Comparison - {sample_name}")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"roc_comparison_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_metric_comparison(performance_table, metric, sample_name, charts_dir):
    data = performance_table[performance_table["sample"] == sample_name].copy()
    data = data.sort_values(metric, ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(data["model"], data[metric], alpha=0.8)
    ax.set_xlabel(metric.upper())
    ax.set_ylabel("Model")
    ax.set_title(f"{metric.upper()} Comparison - {sample_name}")
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"{metric}_comparison_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_decile_comparison(deciles, sample_name, charts_dir):
    data = deciles[deciles["sample"] == sample_name].copy()

    fig, ax = plt.subplots(figsize=(10, 5))

    for model_name in data["model"].unique():
        subset = data[data["model"] == model_name]
        ax.plot(
            subset["risk_decile"].astype(str),
            subset["observed_bad_rate"],
            marker="o",
            linewidth=2.3,
            label=model_name,
        )

    ax.set_xlabel("Risk decile: 1 = highest risk")
    ax.set_ylabel("Observed bad rate")
    ax.set_title(f"Observed Bad Rate by Decile - {sample_name}")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"decile_comparison_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Prepare time-split modelling matrices

In [ ]:
X_train = train[final_woe_features].fillna(0)
y_train = train[TARGET_COL].astype(int)

X_validation = validation[final_woe_features].fillna(0)
y_validation = validation[TARGET_COL].astype(int)

X_oot = oot_test[final_woe_features].fillna(0)
y_oot = oot_test[TARGET_COL].astype(int)

sample_summary = pd.DataFrame({
    "sample": ["train", "validation", "oot_test"],
    "rows": [len(train), len(validation), len(oot_test)],
    "bad_rate": [y_train.mean(), y_validation.mean(), y_oot.mean()],
})

sample_summary

## 6. Benchmark 1: WOE Logistic Regression on time split

It uses:

- historical train sample;
- later validation sample;
- OOT sample.


In [ ]:
model_lr_time = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000,
)

model_lr_time.fit(X_train, y_train)

lr_time_train_pd = model_lr_time.predict_proba(X_train)[:, 1]
lr_time_validation_pd = model_lr_time.predict_proba(X_validation)[:, 1]
lr_time_oot_pd = model_lr_time.predict_proba(X_oot)[:, 1]

lr_time_perf = pd.DataFrame([
    evaluate_model_result(y_train, lr_time_train_pd, "WOE LR - Time split", "train"),
    evaluate_model_result(y_validation, lr_time_validation_pd, "WOE LR - Time split", "validation"),
    evaluate_model_result(y_oot, lr_time_oot_pd, "WOE LR - Time split", "oot_test"),
])

lr_time_perf

## 7. Benchmark 2: WOE Logistic Regression with random stratified split

This benchmark combines all usable time-split datasets and applies random stratified splitting.

The stratification uses both:

- `target_bad`
- issue quarter, if available

This preserves both class balance and vintage mix.  
However, it still mixes historical and future vintages, so its performance can be optimistic.


In [ ]:
full_woe = pd.concat([train, validation, oot_test], axis=0, ignore_index=True)

X_full = full_woe[final_woe_features].fillna(0)
y_full = full_woe[TARGET_COL].astype(int)

if "issue_quarter" in full_woe.columns:
    strata = full_woe["issue_quarter"].astype(str) + "_" + y_full.astype(str)
    strata_counts = strata.value_counts()
    if (strata_counts < 2).any():
        print("Some issue_quarter + target strata have fewer than 2 rows. Falling back to target-only stratification.")
        strata = y_full
else:
    strata = y_full

X_rand_train, X_rand_test, y_rand_train, y_rand_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    stratify=strata,
)

model_lr_random = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000,
)

model_lr_random.fit(X_rand_train, y_rand_train)

lr_random_train_pd = model_lr_random.predict_proba(X_rand_train)[:, 1]
lr_random_test_pd = model_lr_random.predict_proba(X_rand_test)[:, 1]

lr_random_perf = pd.DataFrame([
    evaluate_model_result(y_rand_train, lr_random_train_pd, "WOE LR - Random stratified", "random_train"),
    evaluate_model_result(y_rand_test, lr_random_test_pd, "WOE LR - Random stratified", "random_test"),
])

lr_random_perf

## 8. Benchmark 3: XGBoost challenger on final WOE features

For a clean benchmark, the challenger uses the same final WOE feature set.

This keeps the comparison focused on model algorithm:

```text
Logistic regression vs tree-based gradient boosting
```

rather than mixing algorithm differences with a completely different feature universe.

If `xgboost` is installed, the notebook uses `XGBClassifier`.  
If not, it falls back to `HistGradientBoostingClassifier`, which is a strong sklearn-native gradient boosting alternative.


In [ ]:
try:
    from xgboost import XGBClassifier

    xgb_available = True

    model_xgb = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.80,
        colsample_bytree=0.80,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    challenger_name = "XGBoost - WOE features"

except ImportError:
    xgb_available = False

    model_xgb = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=0.01,
        random_state=42,
    )

    challenger_name = "HistGB - WOE features"

print(f"Using challenger model: {challenger_name}")

model_xgb.fit(X_train, y_train)

xgb_train_pd = model_xgb.predict_proba(X_train)[:, 1]
xgb_validation_pd = model_xgb.predict_proba(X_validation)[:, 1]
xgb_oot_pd = model_xgb.predict_proba(X_oot)[:, 1]

xgb_perf = pd.DataFrame([
    evaluate_model_result(y_train, xgb_train_pd, challenger_name, "train"),
    evaluate_model_result(y_validation, xgb_validation_pd, challenger_name, "validation"),
    evaluate_model_result(y_oot, xgb_oot_pd, challenger_name, "oot_test"),
])

xgb_perf

## 9. Compare AUROC, Gini, KS, Brier and decile separation

In [ ]:
performance_comparison = pd.concat(
    [
        lr_time_perf,
        lr_random_perf,
        xgb_perf,
    ],
    ignore_index=True,
)

performance_comparison

In [ ]:
prediction_tables = []

def append_prediction_table(y_true, pd_pred, model_name, sample_name):
    return pd.DataFrame({
        "model": model_name,
        "sample": sample_name,
        "target": np.asarray(y_true),
        "pd_pred": np.asarray(pd_pred),
    })

prediction_tables.append(append_prediction_table(y_train, lr_time_train_pd, "WOE LR - Time split", "train"))
prediction_tables.append(append_prediction_table(y_validation, lr_time_validation_pd, "WOE LR - Time split", "validation"))
prediction_tables.append(append_prediction_table(y_oot, lr_time_oot_pd, "WOE LR - Time split", "oot_test"))

prediction_tables.append(append_prediction_table(y_rand_train, lr_random_train_pd, "WOE LR - Random stratified", "random_train"))
prediction_tables.append(append_prediction_table(y_rand_test, lr_random_test_pd, "WOE LR - Random stratified", "random_test"))

prediction_tables.append(append_prediction_table(y_train, xgb_train_pd, challenger_name, "train"))
prediction_tables.append(append_prediction_table(y_validation, xgb_validation_pd, challenger_name, "validation"))
prediction_tables.append(append_prediction_table(y_oot, xgb_oot_pd, challenger_name, "oot_test"))

prediction_comparison = pd.concat(prediction_tables, ignore_index=True)

decile_tables = []

for (model_name, sample_name), grp in prediction_comparison.groupby(["model", "sample"]):
    decile_tables.append(
        decile_report(
            grp["target"].values,
            grp["pd_pred"].values,
            model_name,
            sample_name,
        )
    )

decile_comparison = pd.concat(decile_tables, ignore_index=True)

decile_comparison.head(30)

In [ ]:
for sample in ["train", "validation", "oot_test"]:
    plot_roc_comparison(prediction_comparison, sample, CHARTS_DIR)
    plot_metric_comparison(performance_comparison, "auc", sample, CHARTS_DIR)
    plot_metric_comparison(performance_comparison, "gini", sample, CHARTS_DIR)
    plot_metric_comparison(performance_comparison, "ks", sample, CHARTS_DIR)
    plot_decile_comparison(decile_comparison, sample, CHARTS_DIR)

plot_roc_comparison(prediction_comparison, "random_test", CHARTS_DIR)
plot_metric_comparison(performance_comparison, "auc", "random_test", CHARTS_DIR)
plot_metric_comparison(performance_comparison, "gini", "random_test", CHARTS_DIR)
plot_metric_comparison(performance_comparison, "ks", "random_test", CHARTS_DIR)
plot_decile_comparison(decile_comparison, "random_test", CHARTS_DIR)

## 10. Interpretability vs performance 

### WOE Logistic Regression - Time Split

This remains the preferred baseline model because:

- it follows the temporal validation design;
- coefficients are interpretable;
- WOE transformation supports scorecard development;
- it is easier to document, validate and monitor;
- it is more aligned with credit risk model governance.

### WOE Logistic Regression - Random Stratified Split

This benchmark is useful as a sensitivity check.  
It preserves bad-rate and vintage proportions, but it still mixes time periods.

### XGBoost / Gradient Boosting Challenger

The challenger may improve discrimination, but:

- it is less transparent;
- scorecard scaling is less direct;
- governance and explainability requirements are higher;
- it may overfit if not carefully tuned;
- it is better positioned as a challenger rather than the primary baseline model.

For this project, the primary model remains the WOE logistic regression scorecard unless the challenger shows a major and stable improvement on OOT performance.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "model": "WOE LR - Time split",
            "role": "Primary baseline scorecard",
            "strengths": "Interpretable, governance-friendly, time-valid, scorecard-compatible.",
            "limitations": "May underperform nonlinear models.",
            "recommendation": "Preferred baseline model.",
        },
        {
            "model": "WOE LR - Random stratified",
            "role": "Sensitivity benchmark",
            "strengths": "Preserves class and vintage mix; useful optimistic comparison.",
            "limitations": "Temporal mixing; not a true deployment simulation.",
            "recommendation": "Use as supporting benchmark only.",
        },
        {
            "model": challenger_name,
            "role": "Tree-based challenger",
            "strengths": "Can capture nonlinear effects and interactions.",
            "limitations": "Less interpretable; harder governance and scorecard integration.",
            "recommendation": "Use as challenger; compare especially on OOT.",
        },
    ]
)

summary

## 11. Save Excel, pickle and charts

In [ ]:
benchmark_artifacts = {
    "target_col": TARGET_COL,
    "final_woe_features": final_woe_features,
    "sample_summary": sample_summary,
    "performance_comparison": performance_comparison,
    "prediction_comparison": prediction_comparison,
    "decile_comparison": decile_comparison,
    "summary": summary,
    "model_lr_time": model_lr_time,
    "model_lr_random": model_lr_random,
    "model_challenger": model_xgb,
    "challenger_name": challenger_name,
    "xgb_available": xgb_available,
}

with open(OUTPUT_DIR / "model_benchmarking_artifacts.pkl", "wb") as f:
    pickle.dump(benchmark_artifacts, f)

with open(OUTPUT_DIR / "model_lr_time.pkl", "wb") as f:
    pickle.dump(model_lr_time, f)

with open(OUTPUT_DIR / "model_lr_random.pkl", "wb") as f:
    pickle.dump(model_lr_random, f)

with open(OUTPUT_DIR / "model_challenger.pkl", "wb") as f:
    pickle.dump(model_xgb, f)

excel_path = OUTPUT_DIR / "model_benchmarking_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    sample_summary.to_excel(writer, sheet_name="sample_summary", index=False)
    performance_comparison.to_excel(writer, sheet_name="performance", index=False)
    decile_comparison.to_excel(writer, sheet_name="deciles", index=False)
    summary.to_excel(writer, sheet_name="discussion", index=False)
    pd.DataFrame({"final_woe_features": final_woe_features}).to_excel(
        writer, sheet_name="features", index=False
    )

    # Store predictions in separate sheets, limited to avoid very large Excel files.
    prediction_comparison.head(100000).to_excel(
        writer, sheet_name="predictions_sample", index=False
    )

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'model_benchmarking_artifacts.pkl'}")
print(f"- {OUTPUT_DIR / 'model_lr_time.pkl'}")
print(f"- {OUTPUT_DIR / 'model_lr_random.pkl'}")
print(f"- {OUTPUT_DIR / 'model_challenger.pkl'}")
print(f"- {excel_path}")
print(f"- charts saved in {CHARTS_DIR}")

## Conclusions

Three model configurations were benchmarked:

1. WOE Logistic Regression with time-based validation (baseline scorecard model)
2. WOE Logistic Regression with random stratified split
3. XGBoost challenger using the same final WOE feature set

## Key Findings

### Baseline Scorecard Model

The final WOE Logistic Regression achieved:

| Sample     | AUROC | Gini  | KS    | Brier Score |
| ---------- | ----- | ----- | ----- | ----------- |
| Train      | 0.692 | 0.385 | 0.278 | 0.147       |
| Validation | 0.672 | 0.344 | 0.245 | 0.160       |
| OOT        | 0.667 | 0.335 | 0.246 | 0.136       |

The model demonstrates stable performance across Train, Validation and OOT samples. The relatively small decline between Train and OOT performance suggests that the selected variables, WOE transformations and feature selection process generalize reasonably well to future vintages.

The model maintains consistent discriminatory power while preserving full scorecard interpretability and transparency.

### Random Stratified Benchmark

The random stratified split achieved:

| Sample | AUROC | Gini  | KS    | Brier Score |
| ------ | ----- | ----- | ----- | ----------- |
| Train  | 0.688 | 0.377 | 0.271 | 0.148       |
| Test   | 0.688 | 0.377 | 0.273 | 0.148       |

Performance is highly stable because both training and testing samples originate from the same overall population and contain a mixture of historical vintages.

While this approach provides a useful benchmark, it does not reflect a realistic production scenario because future observations are indirectly represented in the training population. As a result, the random split provides a more optimistic estimate of future model performance.

For credit risk modelling, the time-based validation framework remains the preferred methodology because it better simulates real-world model deployment.

### XGBoost Challenger

The XGBoost challenger achieved:

| Sample     | AUROC | Gini  | KS    | Brier Score |
| ---------- | ----- | ----- | ----- | ----------- |
| Train      | 0.699 | 0.397 | 0.286 | 0.146       |
| Validation | 0.678 | 0.357 | 0.256 | 0.159       |
| OOT        | 0.678 | 0.357 | 0.261 | 0.135       |

The challenger consistently outperformed the logistic regression scorecard across all evaluation samples.

Compared with the baseline scorecard:

| Metric      | Logistic Regression OOT | XGBoost OOT |
| ----------- | ----------------------- | ----------- |
| AUROC       | 0.667                   | 0.678       |
| Gini        | 0.335                   | 0.357       |
| KS          | 0.246                   | 0.261       |
| Brier Score | 0.136                   | 0.135       |

The improvement is measurable and consistent across Validation and OOT samples. However, the performance gain remains relatively modest in absolute terms.

Given the additional complexity, reduced interpretability, more demanding governance requirements and more challenging scorecard implementation process, the observed performance improvement alone may not be sufficient to justify replacing the scorecard model within this project.

## Model Selection

The preferred model remains the WOE Logistic Regression scorecard because it provides:

* transparent variable effects;
* interpretable coefficients;
* business-friendly scorecard structure;
* straightforward score scaling;
* easier monitoring and validation;
* stronger governance suitability;
* easier implementation within traditional credit risk environments.

The XGBoost model should be retained as a challenger model for future benchmarking and monitoring purposes.

## Areas for Future Improvement

Potential next steps include:

* hyperparameter optimization of the XGBoost challenger;
* additional feature engineering;
* interaction feature creation;
* segmentation strategies;
* reject inference analysis;
* score calibration and PD mapping;
* profitability-based cutoff optimization;
* population stability and characteristic stability monitoring;
* alternative target definitions and observation windows;
* incorporation of macroeconomic variables.

## Final Assessment

The project demonstrates a complete end-to-end credit risk modelling framework covering:

* data inspection and quality assessment;
* target definition and leakage analysis;
* exploratory data analysis;
* time-based validation design;
* feature engineering;
* WOE transformation and IV analysis;
* correlation and business-rule based variable selection;
* Logistic Regression scorecard development;
* challenger model benchmarking;
* validation and stability assessment;
* score scaling and cutoff analysis;
* governance considerations.

The final model choice should prioritize:

* OOT performance;
* stability;
* interpretability;
* governance suitability;
* scorecard implementation feasibility.

Although the XGBoost challenger delivers a measurable performance improvement, the WOE Logistic Regression scorecard remains the preferred baseline model due to its strong balance between predictive power, stability, explainability and operational usability within a credit risk modelling framework.
